# Rozwiązanie ćwiczenia i wyjaśnienie pojęcia redukcji w obliczeniach

## 1. Przykładowe rozwiązanie ćwiczenia wyliczania NDVI

In [ ]:
import numpy as np
import dask.array as da
import time

# --- KROK 1: Demonstracja OOM w NumPy ---
shape = (365, 4, 3000, 3000)
dtype = np.float32
bytes_needed = np.prod(shape) * np.dtype(dtype).itemsize
gib_needed = bytes_needed / (1024 ** 3)
print(f"Wymagana pamięć dla NumPy: {gib_needed:.2f} GiB")

try:
    arr_numpy = np.empty(shape, dtype=dtype)
    print("Alokacja w NumPy powiodła się (masz maszynę z >64GB wolnego RAM!)")
except (np.core._exceptions._ArrayMemoryError, MemoryError) as e:
    print(f"[SUKCES DEMONSTRACJI OOM] NumPy zgłosił błąd pamięci:\n  -> {e}")

# --- KROK 2: Dask Array Out-of-Core ---
# Chunk (10, 4, 1000, 1000) float32 = 30 * 4 * 1000 * 1000 * 4 B = 480 MB
# Albo mniejszy (5, 4, 1000, 1000) = 76 MB, optymalny dla limitu 1 GiB/worker
chunks = (5, 4, 1000, 1000)
sat_data = da.random.uniform(0.0, 1.0, size=shape, chunks=chunks).astype(dtype)
print("\nInformacje o tablicy Dask:")
display(sat_data)

# --- KROK 3: Wyliczenie NDVI ---
red = sat_data[:, 2, :, :]
nir = sat_data[:, 3, :, :]
# pamiętajmy, że pracujemy z numpy - automatyczny broadcasting i operacje wektorowe
ndvi = (nir - red) / (nir + red + 1e-6)
print(f"Kształt NDVI: {ndvi.shape}, rozmiar w pamięci: {ndvi.nbytes / (1024**3):.2f} GiB")

# --- KROK 4: Obliczenia ze strumieniowaniem ---
start = time.time()
print("\nObliczanie średniego profilu NDVI (365 dni)...")
# redukcja po wymiarze "dzień" czyli do wektora 365 wartości
ndvi_time_series = ndvi.mean(axis=(1, 2)).compute()
print(f"Zakończono w: {time.time() - start:.2f} s. Rozmiar wyniku: {ndvi_time_series.shape}")

start = time.time()
print("Wykrywanie pikseli suszy (NDVI < 0.2)...")
# redukcja do pojedynczej wartości (skalara)
drought_ratio = (ndvi < 0.2).mean().compute()
print(f"Odsetek pikseli z suszą: {drought_ratio * 100:.2f}%, czas: {time.time() - start:.2f} s")

## 2. Mechanizm redukcji w Dask Array na przykładzie obliczeń NDVI

Aby w pełni zrozumieć, dlaczego Dask Array potrafi bez problemu przetworzyć ponad **52 GB** danych wejściowych i wyliczyć statystyki z tablicy NDVI o objętości **13 GB** na maszynie o ograniczonym RAM-ie (np. `memory_limit="1GiB"`), kluczowe jest pojęcie **redukcji** (ang. *reduction*).

---

### 2.1. Czym jest redukcja w obliczeniach wielowymiarowych?

W algebrze tablic wielowymiarowych (*N-dimensional arrays*) **redukcja** to operacja przekształcająca tablicę $N$-wymiarową w tablicę o niższej liczbie wymiarów (np. $(N-1)$, $(N-2)$ lub do pojedynczego skalara) poprzez agregację elementów wzdłuż jednej lub wielu wybranych osi (*axes*).

W typowych operacjach numerycznych redukcjami są m.in.:
* `sum()` (suma)
* `mean()` (średnia arytmetyczna)
* `std()`, `var()` (odchylenie standardowe, wariancja)
* `min()`, `max()` (ekstrema)
* `any()`, `all()` (redukcje logiczne)

---

### 2.2. Transformacja wymiarów: Co stało się z osią barwy (pasm spektralnych)?

Warto zwrócić uwagę na strukturę wymiarów danych. Surowa tablica wejściowa `sat_data` jest **4-wymiarowa (4D)**:
* **Oś 0:** Czas – 365 dni
* **Oś 1:** **Barwa / Pasmo spektralne** – 4 pasma: `[0: Blue, 1: Green, 2: Red, 3: NIR]`
* **Oś 2:** Przestrzeń $Y$ (wysokość) – 3000 pikseli
* **Oś 3:** Przestrzeń $X$ (szerokość) – 3000 pikseli
* **Kształt:** `(365, 4, 3000, 3000)` $\rightarrow$ **52,56 GB**

#### Krok 1: Eliminacja osi barwy poprzez selekcję spektralną (indeksowanie)
Podczas wyznaczania wskaźnika NDVI nie uśredniamy wszystkich pasm matematyczną redukcją typu `mean(axis=1)` (ponieważ średnia ze wszystkich barw reprezentowałaby ogólną jasność/odbicie, a nie stan roślinności). Zamiast tego z osi 1 wybieramy konkretne dwa pasma:
```python
red = sat_data[:, 2, :, :]  # wycięcie pasma Red (indeks 2 z osi 1)
nir = sat_data[:, 3, :, :]  # wycięcie pasma NIR (indeks 3 z osi 1)
ndvi = (nir - red) / (nir + red + 1e-6)
```

**Co dzieje się z osią barwy w tym momencie?**
1. **Redukcja wymiaru przez indeksowanie:** Wycięcie pasm usuwa Oś 1 (wymiar barwy) z tablicy. Z tablicy 4D przechodzimy do dwóch tablic 3D: `red` i `nir`.
2. **Synteza spektralna:** Formuła NDVI syntetyzuje dwa pasma w jeden znormalizowany wskaźnik biofizyczny.
3. **Przesunięcie numeracji osi w tablicy `ndvi`:**
   W wynikowej tablicy `ndvi` osie ulegają przesunięciu w lewo:
   * **Nowa Oś 0:** Czas (365 dni) – dawna Oś 0
   * **Nowa Oś 1:** Przestrzeń $Y$ / wysokość (3000 px) – **dawna Oś 2 z `sat_data`!**
   * **Nowa Oś 2:** Przestrzeń $X$ / szerokość (3000 px) – **dawna Oś 3 z `sat_data`!**

> **Wskazówka dydaktyczna:** Gdybyśmy chcieli wykonać czystą redukcję statystyczną bezpośrednio na osi barwy, użylibyśmy `sat_data.mean(axis=1)`. Operacja ta zredukowałaby 4 pasma do 1 (obliczając średnią luminancję piksela) i również dała tablicę 3D o kształcie `(365, 3000, 3000)`.

---

### 2.3. Dwa rodzaje redukcji statystycznej na tablicy NDVI

Na trójwymiarowej tablicy pośredniej `ndvi` o kształcie `(365, 3000, 3000)` (objętość **13,14 GB**) wykonujemy dwa rodzaje redukcji:

#### A. Redukcja przestrzenna wzdłuż nowych osi 1 i 2 (`axis=(1, 2)`)
```python
ndvi_time_series = ndvi.mean(axis=(1, 2)).compute()
```
* **Co redukujemy:** Nową Oś 1 ($Y$) oraz nową Oś 2 ($X$) – czyli dawne przestrzenne Osie 2 i 3 z surowych danych. Oś 0 (czas) pozostaje nienaruszona.
* **Sens fizyczny:** Dla każdego z 365 dni uśredniamy wszystkie $3000 \times 3000 = 9\,000\,000$ pikseli, uzyskując jedną reprezentatywną wartość NDVI dla całego badanego regionu w danym dniu.
* **Stopień kompresji danych:**
  * Kształt przed redukcją: `(365, 3000, 3000)` $\rightarrow$ **13,14 GB**
  * Kształt po redukcji: `(365,)` $\rightarrow$ $365 \times 4\text{ B} \approx \mathbf{1{,}46\text{ KB}}$!
  * **Współczynnik redukcji:** $9\,000\,000 : 1$.

#### B. Redukcja globalna do pojedynczego skalara (`axis=None`)
```python
drought_ratio = (ndvi < 0.2).mean().compute()
```
* **Co redukujemy:** Wszystkie trzy osie jednocześnie (czas, $Y$, $X$).
* **Sens fizyczny:** Zliczamy, jaki ułamek ze wszystkich 3,28 miliarda obserwacji w całym roku i na całym obszarze spełnia warunek suszy ($\text{NDVI} < 0{,}2$).
* **Stopień kompresji danych:**
  * Przed redukcją: macierz masek logicznych $\rightarrow$ **3,28 GB** (dla typu `bool` / 1 bajt).
  * Po redukcji: pojedyncza liczba zmiennoprzecinkowa $\rightarrow$ **4 bajty**.

---

### 2.4. Różnica: operacje punktowe (*element-wise*) a operacje redukcji

W grafie obliczeniowym zadania występują dwa fundamentalne typy operacji:

| Cecha | Operacje punktowe (*element-wise*) | Operacje redukcji (*reduction*) |
| :--- | :--- | :--- |
| **Przykłady w zadaniu** | `nir - red`, `nir + red`, `(ndvi < 0.2)` | `ndvi.mean(axis=(1, 2))`, `mask.mean()` |
| **Kształt wyniku** | Zachowuje oryginalny kształt tablicy | Usuwa jedną lub więcej osi |
| **Zależności między danymi** | 1:1 (piksel zależy tylko od piksela na tej samej pozycji) | N:1 (wiele pikseli łączy się w jedną wartość) |
| **Objętość danych** | Pozostaje gigantyczna (~13 GB) | Drastycznie spada (do KB lub bajtów) |
| **Wymiana między chunkami** | Brak (chunki liczą się w 100% niezależnie) | Wymaga agregacji wyników cząstkowych |

---

### 2.5. Jak Dask realizuje redukcję pod maską? (Wzorzec Drzewa Redukcji – *Tree Reduction*)

Gdybyśmy spróbowali wykonać redukcję naiwnie:
1. Pobrać wszystkie 13 GB danych do jednego procesu,
2. Wywołać `np.mean()`,
...to natychmiast wywołalibyśmy **OOM**, niszcząc cały zysk z partycjonowania.

Dask realizuje redukcję za pomocą **wielostopniowego drzewa redukcji (*Tree Reduction*)**:

```
[ Chunk 0 (240 MB) ]  --> Oblicz: sum_0, count_0  ---\
[ Chunk 1 (240 MB) ]  --> Oblicz: sum_1, count_1  ----+--> Łączenie cząstkowe: sum_A, count_A ---\
                                                                                                    +--> Finalizacja:
[ Chunk 2 (240 MB) ]  --> Oblicz: sum_2, count_2  ---\                                              |    mean = sum_total / count_total
[ Chunk 3 (240 MB) ]  --> Oblicz: sum_3, count_3  ----+--> Łączenie cząstkowe: sum_B, count_B ---/      (Zwracany wektor: 1.46 KB)
```

#### Etapy wykonania:
1. **Lokalna agregacja w chunkach (*chunk-wise partial reduction*):**
   Każdy worker pobiera wyłącznie swój fragment danych (np. blok $15 \times 1000 \times 1000$ pikseli $\approx 60\text{ MB}$). Worker oblicza dla niego lokalną sumę (`sum`) oraz liczbę zsumowanych elementów (`count`).
   * **Kluczowy moment zwolnienia pamięci:** Oryginalny 60-megabajtowy chunk jest natychmiast usuwany z pamięci RAM (zbierany przez *Garbage Collector*). W pamięci workera pozostają jedynie mikro-wektory sum częściowych!
2. **Drzewiaste scalanie wyników pośrednich (*tree-combine*):**
   Sumy cząstkowe i liczności z poszczególnych chunków są przesyłane pomiędzy węzłami i sumowane parami:
   $$\text{sum}_{\text{total}} = \sum_{i} \text{sum}_i, \quad \text{count}_{\text{total}} = \sum_{i} \text{count}_i$$
3. **Finalizacja (*finalize*):**
   W korzeniu grafu wykonywane jest ostateczne dzielenie: $\text{mean} = \frac{\text{sum}_{\text{total}}}{\text{count}_{\text{total}}}$.

---

### 2.6. Dlaczego to ratuje nas przed OOM?

* **Szczytowe zużycie pamięci (*Peak Memory Footprint*):**
  Zamiast trzymać w pamięci całą tablicę 52 GB (dane surowe) lub 13 GB (NDVI), maksymalne chwilowe zużycie RAM na pojedynczym workerze jest ograniczone do:
  $$\text{RAM}_{\text{worker}} \approx \text{rozmiar aktywnego chunka} + \text{bufor sum pośrednich}$$
  Dla chunka rzędu $150\text{ MB}$, worker potrzebuje zaledwie ok. **$200\text{ MB}$ RAM**. Spokojnie mieści się to w limicie `memory_limit="1GiB"`.
* **Różnica między `.compute()` bez redukcji a z redukcją:**
  * `ndvi.compute()` $\rightarrow$ **BŁĄD / CRASH!** Próba przesłania 13 GB nieskompresowanych danych do pamięci głównej jądra Jupyter Notebook.
  * `ndvi.mean(axis=(1, 2)).compute()` $\rightarrow$ **SUKCES!** Do pamięci jądra wraca wektor 365 liczb (1.46 KB). Obliczenia 52 GB odbyły się w strumieniu.
